vecotr stores : - Chroma, FAISS, Qdrant, Pinecone, Weaviate, Milvus, PostgreSQL (pgvector), MongoDB Atlas Vector Search

In [4]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/Users/yuvrajbhatkariya/data/VScode.C++/MachineLearning/LangChain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7689.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )
docs = [doc1, doc2, doc3, doc4, doc5]

### Create data base : -

In [7]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='chroma_db',
    collection_name='sample'
)

In [8]:
vector_store.add_documents(docs)

['25b238c3-4ca5-422e-bf7f-7cd57e4bbf4b',
 '30549951-1283-4f59-976b-4d432a0545e9',
 'f4592438-c4bf-4d93-a25a-acf6e7dd0105',
 '3751463f-6177-4f5f-97fb-301c12522be3',
 'b78a1ce6-568a-412a-8bac-ee9d08cd5edb']

In [9]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['25b238c3-4ca5-422e-bf7f-7cd57e4bbf4b',
  '30549951-1283-4f59-976b-4d432a0545e9',
  'f4592438-c4bf-4d93-a25a-acf6e7dd0105',
  '3751463f-6177-4f5f-97fb-301c12522be3',
  'b78a1ce6-568a-412a-8bac-ee9d08cd5edb'],
 'embeddings': array([[ 0.00994719,  0.0691433 , -0.05147115, ..., -0.03543334,
          0.01284806,  0.01248283],
        [ 0.00127744,  0.03129857, -0.02375379, ..., -0.00518358,
         -0.03280609,  0.02737712],
        [-0.10265924,  0.02650808,  0.02271502, ..., -0.03359749,
         -0.07984949, -0.01507709],
        [ 0.02123396, -0.02468547, -0.04494365, ..., -0.10995812,
          0.00572562,  0.09915373],
        [ 0.01873973,  0.04382843, -0.04304251, ..., -0.07801621,
         -0.07840677, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [ ]:
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=1
)


[Document(id='3751463f-6177-4f5f-97fb-301c12522be3', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [12]:
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='3751463f-6177-4f5f-97fb-301c12522be3', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693602323532104),
 (Document(id='30549951-1283-4f59-976b-4d432a0545e9', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1493451595306396)]

In [13]:
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='f4592438-c4bf-4d93-a25a-acf6e7dd0105', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436006307601929),
 (Document(id='b78a1ce6-568a-412a-8bac-ee9d08cd5edb', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]

In [14]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [15]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['25b238c3-4ca5-422e-bf7f-7cd57e4bbf4b',
  '30549951-1283-4f59-976b-4d432a0545e9',
  'f4592438-c4bf-4d93-a25a-acf6e7dd0105',
  '3751463f-6177-4f5f-97fb-301c12522be3',
  'b78a1ce6-568a-412a-8bac-ee9d08cd5edb'],
 'embeddings': array([[ 0.00994719,  0.0691433 , -0.05147115, ..., -0.03543334,
          0.01284806,  0.01248283],
        [ 0.00127744,  0.03129857, -0.02375379, ..., -0.00518358,
         -0.03280609,  0.02737712],
        [-0.10265924,  0.02650808,  0.02271502, ..., -0.03359749,
         -0.07984949, -0.01507709],
        [ 0.02123396, -0.02468547, -0.04494365, ..., -0.10995812,
          0.00572562,  0.09915373],
        [ 0.01873973,  0.04382843, -0.04304251, ..., -0.07801621,
         -0.07840677, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [16]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['25b238c3-4ca5-422e-bf7f-7cd57e4bbf4b',
  '30549951-1283-4f59-976b-4d432a0545e9',
  'f4592438-c4bf-4d93-a25a-acf6e7dd0105',
  '3751463f-6177-4f5f-97fb-301c12522be3',
  'b78a1ce6-568a-412a-8bac-ee9d08cd5edb'],
 'embeddings': array([[ 0.00994719,  0.0691433 , -0.05147115, ..., -0.03543334,
          0.01284806,  0.01248283],
        [ 0.00127744,  0.03129857, -0.02375379, ..., -0.00518358,
         -0.03280609,  0.02737712],
        [-0.10265924,  0.02650808,  0.02271502, ..., -0.03359749,
         -0.07984949, -0.01507709],
        [ 0.02123396, -0.02468547, -0.04494365, ..., -0.10995812,
          0.00572562,  0.09915373],
        [ 0.01873973,  0.04382843, -0.04304251, ..., -0.07801621,
         -0.07840677, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo